In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from lanxi_config import LanXiRecorderClient
import time

In [ ]:
client = LanXiRecorderClient(ip_address="169.254.13.140")

In [ ]:
# --- СОЗДАНИЕ ИНТЕРАКТИВНЫХ ВИДЖЕТОВ (ГРУППЫ ПАРАМЕТРОВ) ---
w_rec_name  = widgets.Text(value='test_run_1210', description='Record Name:')
w_bandwidth = widgets.Dropdown(options=['51.2 kHz', '102.4 kHz', '25.6 kHz', '12.8 kHz'], value='51.2 kHz', description='Bandwidth:')
w_max_size  = widgets.Dropdown(options=[('2 GB', 2147483648), ('4 GB', 4294967296), ('1 GB', 1073741824)], value=2147483648, description='Size Limit:')

w_ch_name   = widgets.Text(value='Channel 1', description='Ch 1 Name:')
w_range     = widgets.Dropdown(options=['10 Vpeak', '1 Vpeak', '0.316 Vpeak', '31.6 Vpeak'], value='10 Vpeak', description='Range:')
w_filter    = widgets.Dropdown(options=['7.0 Hz', '0.7 Hz', 'DC', '22.4 Hz'], value='7.0 Hz', description='HP Filter:')
w_ccld      = widgets.Checkbox(value=False, description='Enable CCLD (IEPE) Power')

w_t_number  = widgets.Text(value='4961', description='Transd. No:')
w_t_serial  = widgets.IntText(value=3043249, description='Serial No:')
w_t_sens    = widgets.FloatText(value=0.05663, description='Sensitivity:', step=0.00001)
w_t_unit    = widgets.Dropdown(options=['Pa', 'V', 'm/s²', 'N'], value='Pa', description='Unit:')

btn_apply   = widgets.Button(description='Apply Parameters', button_style='success', icon='check', layout=widgets.Layout(width='200px', margin='10px 0px 0px 0px'))
out_panel   = widgets.Output()

# --- ВЕРСТКА ФОРМЫ ИНТЕРФЕЙСА ---
section_global = widgets.VBox([widgets.HTML("<b>Global Settings</b>"), w_rec_name, w_bandwidth, w_max_size])
section_signal = widgets.VBox([widgets.HTML("<b>Electrical Channel 1</b>"), w_ch_name, w_range, w_filter, w_ccld])
section_trans  = widgets.VBox([widgets.HTML("<b>Transducer Properties</b>"), w_t_number, w_t_serial, w_t_sens, w_t_unit])

form_ui = widgets.VBox([
    widgets.HBox([section_global, section_signal, section_trans], layout=widgets.Layout(justify_content='space-between', padding='10px')),
    btn_apply
])

def on_apply_clicked(b):
    with out_panel:
        clear_output()
        
        config_payload = {
            "name": w_rec_name.value,
            "fileFormat": "wav",
            "maxSize": w_max_size.value,
            "recordingMode": "Single",
            "channels": [
                {
                    "channel": 1,
                    "name": w_ch_name.value,
                    "enabled": True,
                    "range": w_range.value,
                    "filter": w_filter.value,
                    "floating": False,
                    "ccld": w_ccld.value,
                    "destinations": ["sd"],
                    "bandwidth": w_bandwidth.value,
                    "transducer": {
                        "requiresCcld": w_ccld.value,
                        "requires200V": False,
                        "sensitivity": w_t_sens.value,
                        "serialNumber": w_t_serial.value,
                        "type": {"model": "", "number": w_t_number.value, "prefix": "", "variant": ""},
                        "unit": w_t_unit.value
                    }
                }
            ]
        }
        
        print(f"{int(time.time())}: Packaging current UI inputs into atomic config...")
        
        # Сброс и отправка через формализованную FSM
        client.stop_and_close()
        time.sleep(1)
        
        if client.apply_configuration(config_payload):
            print(f"{int(time.time())}: Success: Parameters applied. Device is in 'RecorderStreaming' state.")
        else:
            print(f"{int(time.time())}: Error: FSM rejected transition.")

btn_apply.on_click(on_apply_clicked)
display(form_ui, out_panel)

Output()

In [ ]:
# Write config to SD
if client.start_recording():
    time.sleep(2)
    client.stop_recording()

client.stop_and_close()

1792147870: Starting hardware recording to SD card...
1792147872: Stopping active recording...
